In [1]:
import zarr
import re
import os
import gc
import time

import numpy as np
import scanpy as sc
import pandas as pd

In [2]:
def sanitize_key(key):
    return re.sub(r'[^\w]', '_', str(key)).strip('_')

**Compare HVGs**

In [3]:
adata = sc.read_h5ad('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/lucav2/core_250130.h5ad')
adata.obs['cell_id'] = adata.obs.index.astype(str)
adata.obs.reset_index(drop=True, inplace=True)
for col in ['cell_type','dataset','origin']:
    adata.obs[col] = adata.obs[col].astype(str)

adata.obs['comb_key'] = adata.obs.apply(
    lambda r: "-".join([
        sanitize_key(r['cell_type']),
        sanitize_key(r['dataset']),
        sanitize_key(r['origin'])
    ]),
    axis=1
)

In [22]:
for split in ['split01', 'split02', 'split03']:
    train_combinations = zarr.open_group(f'/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_counts/{split}/split_metadata.zarr', mode='r').attrs["train_combinations"]
    adata.obs["is_train"] = adata.obs.apply(lambda x: x.comb_key in train_combinations, axis=1)
    
    tmp = adata[adata.obs.is_train].copy()
    sc.pp.highly_variable_genes(tmp, flavor="seurat_v3", batch_key="dataset", n_top_genes=5983, layer="count")
    
    train_hvgs = set(tmp.var[tmp.var.highly_variable]["feature_name"].to_list())
    hvgs = set(zarr.open_group('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/luca_counts/comb_w_obs.zarr').attrs['var_names'])
    print(f"{split}: {len(train_hvgs.intersection(hvgs)) / len(hvgs)}")

    del tmp, train_hvgs, hvgs
    gc.collect()
    time.sleep(10)

/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


split01: 0.6095604211933813


/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


split02: 0.6740765502256393


/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


split03: 0.6398128029416681
